In [ ]:
"""
Step 04: Timestamp Context (TSCTX) Data Collection

Objective:

    - Add the context that the session-by-session snapshot pipeline (steps 01 / 02 / 03) cannot see:
      time of day, previous sessions, overnight gap, relative volume, distance from the all-time high.
    - One row per minute bar. Every value of the row at bar t uses only bars <= t (verified in tests/run_tests.py).
    - VIX is not included (external data source required).

The rows depend only on past data, so existing files never need to be regenerated when new data is appended
(unless the raw data of past sessions is corrected).
"""
None

In [ ]:
# IMPORT GENERAL PACKAGES
import pandas as pd
import numpy as np
# IMPORT PLOTLY EXPRESS
import plotly.graph_objects as go
from plotly.subplots import make_subplots
# IMPORT THE SHARED CONFIGURATION
from so import config
# IMPORT TRANSFORM OHLCV DATA FUNCTIONS
from so.features.ohlcv_data_utils import format_ohlcv_pdf
# IMPORT PLOT OHLCV FEATURES FUNCTIONS
from so.features.plot_ohlcv_utils import fig_add_ohlcv
# IMPORT MARKET DATETIME FUNCTIONS
from so.core.datetime_utils import ny_tz, get_date_pdf, get_date_range_market_schedule_pdf
# IMPORT LOCAL FILE MANAGEMENT FUNCTIONS
from so.core.local_file_management import get_path_file_list, \
                                    read_pdf_from_csv_file_path_list, \
                                    read_date_range_csv_files_from_path, \
                                    generate_func_data_date_list, \
                                    get_missing_date_list
# IMPORT BARRIER LABEL FUNCTIONS (MARKET OPEN DICTIONARY)
from so.features.barrier_labels import get_date_market_open_ts_dict
# IMPORT CONTEXT FEATURE FUNCTIONS
from so.features.context_features import add_cum_max_col, get_session_summary_pdf, get_relative_volume_reference_pdf, get_date_TSCTX_pdf
# IMPORT PATHS
from so import paths

In [ ]:
# CALL FUNCTION TO GET ALL FILES IN PATH
rawzone_data_file_path_list = get_path_file_list(paths.LOCAL_OHLCV_DATA_FILE_PATH_STR)[:]
# CALL FUNCTION TO READ PDF FROM PATH LIST
raw_ohlcv_pdf = read_pdf_from_csv_file_path_list(rawzone_data_file_path_list)
# CORRECT THE TIMESTAMP DATE COLUMN
raw_ohlcv_pdf["timestamp"] = pd.to_datetime(raw_ohlcv_pdf.timestamp, utc=True).dt.tz_convert(ny_tz)
raw_ohlcv_pdf["date"] = raw_ohlcv_pdf["timestamp"].dt.date
# CALL FUNCTION TO COLLECT DATA AND SORT BY TIMESTAMP
complete_ohlcv_pdf = format_ohlcv_pdf(raw_ohlcv_pdf.copy()).sort_values("timestamp").reset_index(drop=True)
# DISPLAY INFORMATION
print(f"min timestamp:\t{complete_ohlcv_pdf.timestamp.min()}")
print(f"max timestamp:\t{complete_ohlcv_pdf.timestamp.max()}")
# PREVIEW DATAFRAME
complete_ohlcv_pdf

In [ ]:
# COLLECT THE START AND END DATE
start_date_str, end_date_str = str(complete_ohlcv_pdf["date"].min()), str(complete_ohlcv_pdf["date"].max())
# CALL FUNCTION TO GET THE MARKET SCHEDULE OF THE DATA RANGE
market_schedule_pdf = get_date_range_market_schedule_pdf(start_date_str, end_date_str)
# CREATE THE DATE TO MARKET OPEN DICTIONARY
date_market_open_ts_dict = get_date_market_open_ts_dict(market_schedule_pdf)
# CREATE THE DATE TO (MARKET OPEN, MARKET CLOSE) DICTIONARY (THE CLOSE IS THE LAST BAR TIMESTAMP)
date_market_schedule_dict = dict(zip(pd.to_datetime(market_schedule_pdf["date"]).dt.date,
                                     zip(market_schedule_pdf["market_open_ts"], market_schedule_pdf["market_close_ts"])))
# PREVIEW DATAFRAME
market_schedule_pdf

In [ ]:
# CALL FUNCTION TO ADD THE RUNNING MAXIMUM HIGH (SAME DEFINITION AS THE STEP03 TSSEG CUM MAX)
complete_ohlcv_pdf = add_cum_max_col(complete_ohlcv_pdf)
# CALL FUNCTION TO GET THE SESSION SUMMARY
session_summary_pdf = get_session_summary_pdf(complete_ohlcv_pdf)
# CALL FUNCTION TO GET THE RELATIVE VOLUME REFERENCE (MEAN VOLUME OF THE SAME MINUTE OVER THE PREVIOUS SESSIONS)
relative_volume_reference_pdf = get_relative_volume_reference_pdf(complete_ohlcv_pdf, date_market_open_ts_dict)
# PREVIEW DATAFRAME
session_summary_pdf

In [ ]:
# FUNCTION: GENERATE THE DATE TIMESTAMP CONTEXT (TSCTX) DATAFRAME
def get_date_TSCTX_data_pdf(date_str_in):
    # CALL FUNCTION TO GENERATE THE DATE TSCTX DATAFRAME
    return get_date_TSCTX_pdf(complete_ohlcv_pdf, session_summary_pdf, relative_volume_reference_pdf, date_market_schedule_dict, date_str_in)

# DEFINE SAMPLE DATE
sample_date_str = str(complete_ohlcv_pdf["date"].iloc[-1])
# CALL FUNCTION TO GENERATE THE SAMPLE DATE TSCTX DATAFRAME
sample_date_TSCTX_pdf = get_date_TSCTX_data_pdf(sample_date_str)
# PREVIEW DATAFRAME
sample_date_TSCTX_pdf

In [ ]:
# FUNCTION: PLOT THE DATE TIMESTAMP CONTEXT (TSCTX) FEATURES
def plot_TSCTX_features_pdf(ohlcv_pdf_in, TSCTX_pdf_in, width_in=1100, height_in=800):
    # CREATE SUBPLOTS: 1 COLUMN, 3 ROWS, SHARED X-AXIS
    fig = make_subplots(rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.05,
                        subplot_titles=("OHLCV", "Intraday Return & Prev Day High/Low Distance", "Relative Volume"))
    # PLOT CANDLESTICKS
    fig_add_ohlcv(fig, ohlcv_pdf_in, row_in=1, col_in=1)
    # PLOT THE RETURN FEATURES
    for col_str in ["intraday_return_pct", "prev_day_high_dist_pct", "prev_day_low_dist_pct"]:
        fig.add_trace(go.Scatter(x=TSCTX_pdf_in["timestamp"], y=TSCTX_pdf_in[col_str], mode="lines", name=col_str), row=2, col=1)
    # PLOT THE RELATIVE VOLUME
    fig.add_trace(go.Scatter(x=TSCTX_pdf_in["timestamp"], y=TSCTX_pdf_in["relative_volume"], mode="lines", name="relative_volume"), row=3, col=1)
    # UPDATE LAYOUT (DARK MODE)
    fig.update_layout(template="plotly_dark", width=width_in, height=height_in, xaxis_rangeslider_visible=False,
                      font=dict(color="white"), title="Timestamp Context (TSCTX) Features")
    # SHOW FIGURE
    fig.show()

# CALL FUNCTION TO PLOT THE SAMPLE DATE TSCTX FEATURES
plot_TSCTX_features_pdf(get_date_pdf(complete_ohlcv_pdf, sample_date_str), sample_date_TSCTX_pdf)

In [ ]:
# GET MISSING DATE LIST
missing_date_list = get_missing_date_list(paths.LOCAL_TSCTX_DATA_FILE_PATH_STR, start_date_str, end_date_str)
# PREVIEW LIST
missing_date_list[:10]

In [ ]:
# CALL FUNCTION TO PROCESS DATE LIST
generate_func_data_date_list(get_date_TSCTX_data_pdf, missing_date_list, paths.LOCAL_TSCTX_DATA_FILE_PATH_STR, "TSCTX_data", "I")

In [ ]:
# CALL FUNCTION TO READ THE LAST MONTH OF TSCTX DATA
TSCTX_data_pdf = read_date_range_csv_files_from_path(paths.LOCAL_TSCTX_DATA_FILE_PATH_STR, str((pd.to_datetime(end_date_str) - pd.DateOffset(months=1)).date()), end_date_str)
# DISPLAY THE NULL COUNTS
display(TSCTX_data_pdf.isna().sum().reset_index().rename(columns={"index": "column", 0: "null_count"}))
# PREVIEW DATAFRAME
TSCTX_data_pdf